# 🧠 TrueRead v3 — Fixed Training Pipeline

> **Architecture:** Mini-ResNet with Residual Connections  
> **Framework:** TensorFlow / Keras &nbsp;|&nbsp; **GPU:** Google Colab T4  
> **Output:** `trueread_model.onnx` · `trueread_model.tflite` · `class_mapping.json`

## What changed from v2 and why

| Problem in v2 | Root Cause | Fix in v3 |
|---|---|---|
| ba→bha, ma→bha wrong predictions in Unity | Model trained on clean data was 99.9% confident on every prediction — even wrong ones | **Label Smoothing (ε=0.1)** — model learns calibrated confidence, not blind certainty |
| Model fails on real handwriting | 99% accuracy on clean test ≠ robust to real-world variation. Model memorised pixel patterns, not stroke structure | **Stronger augmentation** — stroke width variation, perspective, brightness simulate real handwriting |
| Training took hours | `tf.py_function` + albumentations ran per-image on CPU, breaking the TF graph | **GPU-native augmentation** via `tf.keras.layers` — stays in TF graph, runs on T4 GPU |

---

| Cell | Purpose |
|------|---------|
| 1 | Install dependencies |
| 2 | Imports |
| 3 | ⚙️ Configuration |
| 4 | Mount Drive & verify |
| 5 | Class mapping |
| 6 | **Dataset pipeline (GPU augmentation — faster)** |
| 7 | Mini-ResNet architecture |
| 8 | **Focal Loss + Label Smoothing** |
| 9 | Training |
| 10 | Results & evaluation |
| 11 | Inference preprocessing (unchanged from v2) |
| 12 | Preprocessing visualizer (unchanged from v2) |
| 13 | Single-image inference with confidence gate |
| 14 | Unity confidence threshold — C# snippet |
| 15 | TFLite export |
| 16 | ONNX export |
| 17 | Save to Drive |

## 🔧 Cell 1 — Install Dependencies

> Note: `albumentations` is no longer required. Augmentation now runs natively on GPU.

In [ ]:
# albumentations removed — augmentation is now GPU-native via tf.keras.layers
!pip install 'tensorflow==2.15.0' tf2onnx onnx -q
print('✅ Dependencies installed')

## 📦 Cell 2 — Imports

In [ ]:
import os, re, json, shutil
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt
import cv2

gpus = tf.config.list_physical_devices('GPU')
print(f'TensorFlow : {tf.__version__}')
print(f'GPU        : {gpus}')
if not gpus:
    print('⚠️  No GPU! Go to Runtime → Change runtime type → T4 GPU')

## ⚙️ Cell 3 — Configuration
> **Only edit `DRIVE_DATASET_PATH`.**  
> Point it to the folder on Google Drive containing `train/` and `test/` subdirectories.

In [ ]:
DRIVE_DATASET_PATH = '/content/drive/MyDrive/TrueRead_Dataset'  # <-- EDIT THIS

IMG_SIZE      = 64
NUM_CLASSES   = 46
BATCH_SIZE    = 64
EPOCHS        = 60
LEARNING_RATE = 1e-3
VAL_SPLIT     = 0.15

TRAIN_DIR = os.path.join(DRIVE_DATASET_PATH, 'train')
TEST_DIR  = os.path.join(DRIVE_DATASET_PATH, 'test')

print(f'Dataset root : {DRIVE_DATASET_PATH}')
print(f'Image size   : {IMG_SIZE}×{IMG_SIZE}  |  Classes: {NUM_CLASSES}')
print(f'Batch size   : {BATCH_SIZE}  |  Max epochs: {EPOCHS}')

## 📁 Cell 4 — Mount Google Drive & Verify Dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

assert os.path.isdir(TRAIN_DIR), f'❌ Train dir not found: {TRAIN_DIR}'
assert os.path.isdir(TEST_DIR),  f'❌ Test dir not found:  {TEST_DIR}'

train_folders = [f for f in os.listdir(TRAIN_DIR) if os.path.isdir(os.path.join(TRAIN_DIR, f))]
test_folders  = [f for f in os.listdir(TEST_DIR)  if os.path.isdir(os.path.join(TEST_DIR,  f))]

print(f'✅ Train: {len(train_folders)} class folders')
print(f'✅ Test : {len(test_folders)} class folders')
print(f'\nSample folders:')
for f in sorted(train_folders)[:5]:
    n = len(os.listdir(os.path.join(TRAIN_DIR, f)))
    print(f'  {f:<32} ({n} images)')

if len(train_folders) != NUM_CLASSES:
    print(f'⚠️  Expected {NUM_CLASSES}, found {len(train_folders)}. Update NUM_CLASSES in Cell 3.')

## 🗂️ Cell 5 — Class Mapping
Produces `class_mapping.json` — the shared index-to-label dictionary used by both the model and Unity.

In [ ]:
def natural_sort_key(name):
    group = 0 if name.startswith('character_') else (1 if name.startswith('digit_') else 2)
    nums  = re.findall(r'\d+', name)
    return (group, int(nums[0]) if nums else 999)

def extract_label(name):
    if name.startswith('character_'):
        parts = name.split('_', 2)
        return parts[2] if len(parts) > 2 else name
    return name

def build_class_mapping(train_dir):
    folders = [f for f in os.listdir(train_dir) if os.path.isdir(os.path.join(train_dir, f))]
    sorted_folders = sorted(folders, key=natural_sort_key)
    assert len(sorted_folders) == NUM_CLASSES, f'Found {len(sorted_folders)} classes, expected {NUM_CLASSES}'
    mapping = {
        str(idx): {'index': idx, 'folder_name': f, 'label': extract_label(f),
                   'type': 'digit' if f.startswith('digit_') else 'character'}
        for idx, f in enumerate(sorted_folders)
    }
    return mapping, sorted_folders

CLASS_MAPPING, SORTED_FOLDERS = build_class_mapping(TRAIN_DIR)

with open('class_mapping.json', 'w', encoding='utf-8') as f:
    json.dump(CLASS_MAPPING, f, ensure_ascii=False, indent=2)

print('✅ class_mapping.json saved\n')
print(f'{"Idx":<5} {"Folder":<32} {"Label":<18} {"Type"}')
print('─' * 70)
for v in CLASS_MAPPING.values():
    print(f"{v['index']:<5} {v['folder_name']:<32} {v['label']:<18} {v['type']}")

## 📊 Cell 6 — Dataset Pipeline (GPU-Native Augmentation)

### Why the augmentation changed

**v2 problem:** `tf.py_function` breaks the TensorFlow execution graph and sends each image
to CPU for albumentations processing. With 78,000 training images this is the biggest bottleneck.

**v3 fix:** `tf.keras.layers` augmentation layers stay inside the TF graph and run
entirely on the T4 GPU. **Estimated speedup: 2–3×.**

### Why the augmentation is stronger than v2

v2 had 99% test accuracy but failed on real handwriting. The model memorised
perfectly clean pixel patterns. To fix this, we add:

| New augmentation | What it simulates |
|---|---|
| `RandomZoom` ±8% | Different writing distances from camera |
| `RandomTranslation` ±5% | Character not perfectly centred after preprocessing |
| Morphological dilation/erosion | Thicker or thinner pen strokes (most important for ba/bha confusion) |
| `GaussianNoise` σ=0.03 | Residual sensor noise after preprocessing |

**The training images themselves are still clean** — augmentation applies only during training,
not to the stored files. The inference preprocessing pipeline is unchanged.

In [ ]:
# ── Normalization (applied to all splits) ──────────────────────────────────
def normalize(image, label):
    """Cast uint8 [0,255] → float32 [0.0, 1.0]. No other transformation."""
    return tf.cast(image, tf.float32) / 255.0, label


# ── GPU-native augmentation model ──────────────────────────────────────────
# All layers below run on the GPU inside the TF graph — no py_function needed.
gpu_augmentation = tf.keras.Sequential([
    # Rotation: ±7 degrees. factor = degrees / 360.
    # fill_mode='constant', fill_value=0 fills empty corners with black.
    layers.RandomRotation(
        factor=7/360,
        fill_mode='constant',
        fill_value=0.0,
        seed=42
    ),
    # Zoom: ±8% — simulates the character being written at different distances.
    layers.RandomZoom(
        height_factor=(-0.08, 0.08),
        width_factor=(-0.08, 0.08),
        fill_mode='constant',
        fill_value=0.0,
        seed=42
    ),
    # Translation: ±5% — character not perfectly centred after preprocessing.
    layers.RandomTranslation(
        height_factor=0.05,
        width_factor=0.05,
        fill_mode='constant',
        fill_value=0.0,
        seed=42
    ),
    # Gaussian noise: simulates residual camera noise post-preprocessing.
    layers.GaussianNoise(stddev=0.03),
], name='gpu_augmentation')


def apply_augmentation(image, label):
    """
    Run gpu_augmentation + morphological stroke-width variation.
    Called only on the training split.
    """
    image = gpu_augmentation(image, training=True)

    # ── Morphological dilation/erosion (stroke width simulation) ──────────
    # This is the most important augmentation for fixing ba/bha confusion.
    # Different writers press the pen with different force, producing
    # thicker or thinner strokes. A model trained only on clean uniform
    # strokes has never seen this variation.
    #
    # Implementation: use max-pooling as morphological dilation (thickens strokes)
    # and average-pooling + threshold as erosion (thins strokes).
    def morph_op(img):
        img4d = img[tf.newaxis]  # (1, H, W, 1) — required by pooling ops
        r = tf.random.uniform([], 0, 3, dtype=tf.int32)  # 0=none, 1=dilate, 2=erode

        def dilate():  # thicken strokes
            return tf.nn.max_pool2d(img4d, ksize=2, strides=1, padding='SAME')[0]

        def erode():   # thin strokes
            pooled = tf.nn.avg_pool2d(img4d, ksize=2, strides=1, padding='SAME')[0]
            return tf.cast(pooled > 0.5, tf.float32)  # re-binarize after blur

        def identity():
            return img

        return tf.switch_case(r, branch_fns={0: identity, 1: dilate, 2: erode})

    image = tf.py_function(morph_op, [image], tf.float32)
    image.set_shape([IMG_SIZE, IMG_SIZE, 1])  # restore shape after py_function
    return image, label


# ── Load datasets ───────────────────────────────────────────────────────────
_shared = dict(
    image_size  = (IMG_SIZE, IMG_SIZE),
    color_mode  = 'grayscale',
    class_names = SORTED_FOLDERS,  # locks index order to class_mapping.json
    label_mode  = 'categorical',
    batch_size  = None,
    seed        = 42,
)

train_raw = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR, shuffle=True,
    validation_split=VAL_SPLIT, subset='training', **_shared)
val_raw   = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR, shuffle=False,
    validation_split=VAL_SPLIT, subset='validation', **_shared)
test_raw  = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR, shuffle=False, **_shared)

AUTOTUNE = tf.data.AUTOTUNE

# Training: normalize → augment → batch → prefetch
train_ds = (
    train_raw
    .map(normalize,          num_parallel_calls=AUTOTUNE)
    .map(apply_augmentation, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)
# Validation & test: normalize only
val_ds = (
    val_raw
    .map(normalize, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)
test_ds = (
    test_raw
    .map(normalize, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

n_train = sum(1 for _ in train_raw)
n_val   = sum(1 for _ in val_raw)
n_test  = sum(1 for _ in test_raw)
print(f'✅ Loaded — Train: {n_train:,}  |  Val: {n_val:,}  |  Test: {n_test:,}')

for xb, yb in train_ds.take(1):
    print(f'   Batch X : {xb.shape}  dtype={xb.dtype}')
    print(f'   Batch Y : {yb.shape}  dtype={yb.dtype}')

### 👁️ Verify Augmentation (run this to visually check augmented samples)
Run the cell below to see 8 randomly augmented versions of a training image.
**Strokes should be recognisable but with varying thickness, slight rotation, and minor noise.**

In [ ]:
# Pull one clean image from the training set
for sample_img, sample_lbl in train_raw.take(1):
    pass
clean = tf.cast(sample_img, tf.float32) / 255.0  # (64, 64, 1)
label_idx = tf.argmax(sample_lbl).numpy()
label_name = CLASS_MAPPING[str(label_idx)]['label']

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
fig.suptitle(f'Augmented samples of class: "{label_name}"  (index {label_idx})',
             fontsize=12, fontweight='bold')

axes[0][0].imshow(clean[:, :, 0], cmap='gray')
axes[0][0].set_title('Original (clean)', fontsize=9)
axes[0][0].axis('off')

for i, ax in enumerate(axes.flatten()[1:], 1):
    aug_img, _ = apply_augmentation(clean, sample_lbl)
    ax.imshow(aug_img[:, :, 0], cmap='gray')
    ax.set_title(f'Augmented #{i}', fontsize=9)
    ax.axis('off')

plt.tight_layout()
plt.show()
print('✅ If strokes are distorted beyond recognition, reduce augmentation strength in Cell 6.')

## 🏗️ Cell 7 — Mini-ResNet Architecture (unchanged from v2)

The architecture is the same as v2. The accuracy problem was not architectural —
it was overconfidence and lack of domain variation in training. No changes needed here.

In [ ]:
def residual_block(x, filters, stride=1, name='rb'):
    """
    Standard ResNet residual block.
    Main path: Conv→BN→ReLU→Conv→BN
    Shortcut:  identity (or 1×1 projection if shape changes)
    Output:    Add(main, shortcut) → ReLU
    """
    shortcut = x
    x = layers.Conv2D(filters, 3, strides=stride, padding='same', use_bias=False, name=f'{name}_c1')(x)
    x = layers.BatchNormalization(name=f'{name}_bn1')(x)
    x = layers.ReLU(name=f'{name}_r1')(x)
    x = layers.Conv2D(filters, 3, strides=1,      padding='same', use_bias=False, name=f'{name}_c2')(x)
    x = layers.BatchNormalization(name=f'{name}_bn2')(x)

    if stride != 1 or int(shortcut.shape[-1]) != filters:
        shortcut = layers.Conv2D(filters, 1, strides=stride, padding='same', use_bias=False, name=f'{name}_sc_c')(shortcut)
        shortcut = layers.BatchNormalization(name=f'{name}_sc_bn')(shortcut)

    x = layers.Add(name=f'{name}_add')([x, shortcut])
    x = layers.ReLU(name=f'{name}_r2')(x)
    return x


def build_trueread_model(num_classes=NUM_CLASSES, img_size=IMG_SIZE):
    inputs = keras.Input(shape=(img_size, img_size, 1), name='input')
    x = layers.Conv2D(32, 3, strides=1, padding='same', use_bias=False, name='stem_conv')(inputs)
    x = layers.BatchNormalization(name='stem_bn')(x)
    x = layers.ReLU(name='stem_relu')(x)
    x = residual_block(x, 32,  stride=1, name='rb1')  # 64×64×32
    x = residual_block(x, 64,  stride=2, name='rb2')  # 32×32×64
    x = residual_block(x, 128, stride=2, name='rb3')  # 16×16×128
    x = residual_block(x, 128, stride=1, name='rb4')  # 16×16×128
    x = residual_block(x, 256, stride=2, name='rb5')  #  8×8×256
    x = layers.GlobalAveragePooling2D(name='gap')(x)
    x = layers.Dropout(0.4, name='dropout')(x)
    outputs = layers.Dense(num_classes, activation='softmax', name='predictions')(x)
    return keras.Model(inputs, outputs, name='TrueRead_MiniResNet_v3')


model = build_trueread_model()
model.summary(line_length=80)
print(f'\n✅ Parameters: {model.count_params():,}')

## 📉 Cell 8 — Focal Loss with Label Smoothing

### The core fix for wrong predictions in your Unity app

**Why the model was confidently wrong:**  
v2 trained with hard one-hot labels `[0, 0, 1, 0, 0, ...]`.  
The model learned to push the correct class probability to 0.99+,  
and suppressed all others to nearly 0.  
On a real handwritten `ba` that looks slightly like `bha`,  
the model still outputs 0.92 confidence — for the wrong class.

**What Label Smoothing does:**  
Instead of training on `[0, 0, 1, 0, ...]`,  
it trains on `[0.002, 0.002, 0.896, 0.002, ...]` (smoothed by ε=0.1).  

The model learns: *"I should be confident, but not completely certain."*  
On a genuinely ambiguous input, it now outputs 0.55 instead of 0.94 —  
low enough to trigger the confidence gate in your Unity app.

**Effect on accuracy:** Label smoothing slightly lowers raw test accuracy (e.g. 99% → 92–95%),  
but dramatically improves **real-world reliability** — which is the actual goal.

```
Smoothed label formula:
  y_smooth = y_hard × (1 - ε) + ε / num_classes
  where ε = 0.1, num_classes = 46

  Correct class:   1.0 × 0.9  + 0.1/46 = 0.9022
  Other classes:   0.0 × 0.9  + 0.1/46 = 0.0022
```

In [ ]:
LABEL_SMOOTHING = 0.1   # ε — how much probability to redistribute to wrong classes


class FocalLossWithSmoothing(keras.losses.Loss):
    """
    Focal Loss + Label Smoothing.

    Focal Loss prevents easy examples from dominating training,
    forcing the model to focus on hard look-alike character pairs.

    Label Smoothing prevents overconfidence, producing calibrated
    probability outputs that reflect genuine uncertainty on ambiguous inputs.

    Args:
        gamma:   Focal exponent. γ=2 is the canonical value.
        alpha:   Class-balance weight.
        epsilon: Label smoothing factor. 0 = no smoothing (v2 behaviour).
                 0.1 = redistribute 10% of probability mass to wrong classes.
    """
    def __init__(self, gamma=2.0, alpha=0.25, epsilon=0.1, num_classes=NUM_CLASSES, **kwargs):
        super().__init__(**kwargs)
        self.gamma       = gamma
        self.alpha       = alpha
        self.epsilon     = epsilon
        self.num_classes = num_classes

    def call(self, y_true, y_pred):
        # ── Label smoothing ──────────────────────────────────────────────
        y_smooth = y_true * (1.0 - self.epsilon) + (self.epsilon / self.num_classes)

        # ── Focal Loss ───────────────────────────────────────────────────
        y_pred   = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)
        ce       = -y_smooth * tf.math.log(y_pred)
        focal_wt = self.alpha * y_smooth * tf.pow(1.0 - y_pred, self.gamma)
        return tf.reduce_mean(tf.reduce_sum(focal_wt * ce, axis=-1))

    def get_config(self):
        return {**super().get_config(),
                'gamma': self.gamma, 'alpha': self.alpha,
                'epsilon': self.epsilon, 'num_classes': self.num_classes}


model.compile(
    optimizer = keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss      = FocalLossWithSmoothing(gamma=2.0, alpha=0.25, epsilon=LABEL_SMOOTHING),
    metrics   = ['accuracy']
)
print(f'✅ Compiled — Adam + FocalLoss(γ=2, α=0.25) + LabelSmoothing(ε={LABEL_SMOOTHING})')

## 🏋️ Cell 9 — Training

> ⏱️ Estimated training time on T4 GPU: **~30–50 minutes** (down from several hours in v2)

The speed improvement comes from GPU-native augmentation — no CPU bottleneck.

| Callback | Trigger | Action |
|---|---|---|
| `EarlyStopping` | val_loss flat for 12 epochs | Stop + restore best weights |
| `ReduceLROnPlateau` | val_loss flat for 6 epochs | Halve learning rate (floor: 1e-6) |
| `ModelCheckpoint` | Every epoch | Save to Drive if val_accuracy improved |

In [ ]:
CHECKPOINT_PATH = os.path.join(DRIVE_DATASET_PATH, 'best_trueread_v3.keras')

callbacks = [
    keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=12,
        restore_best_weights=True, verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=6, min_lr=1e-6, verbose=1
    ),
    keras.callbacks.ModelCheckpoint(
        CHECKPOINT_PATH, monitor='val_accuracy', save_best_only=True, verbose=1
    ),
]

print(f'Checkpoint path: {CHECKPOINT_PATH}\n')
history = model.fit(
    train_ds, validation_data=val_ds,
    epochs=EPOCHS, callbacks=callbacks, verbose=1
)
print('\n✅ Training complete')

## 📈 Cell 10 — Results & Evaluation

⚠️ **Expected: test accuracy will be lower than v2's 99%** (e.g. 88–94%).  
This is correct and intentional. Label smoothing prevents the model from memorising clean data.  
The real-world performance in your app will be much better.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('TrueRead v3 — Training Results', fontsize=14, fontweight='bold')
ep = range(1, len(history.history['loss']) + 1)

ax1.plot(ep, history.history['loss'],     label='Train', color='royalblue', lw=2)
ax1.plot(ep, history.history['val_loss'], label='Val',   color='tomato',    lw=2, ls='--')
ax1.set_title('Focal Loss (smoothed)'); ax1.set_xlabel('Epoch')
ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(ep, history.history['accuracy'],     label='Train', color='royalblue', lw=2)
ax2.plot(ep, history.history['val_accuracy'], label='Val',   color='tomato',    lw=2, ls='--')
ax2.axhline(0.75, color='limegreen', ls=':', lw=1.5, label='75% floor')
ax2.set_title('Accuracy'); ax2.set_xlabel('Epoch')
ax2.legend(); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

test_loss, test_acc = model.evaluate(test_ds, verbose=0)
train_acc_final = history.history['accuracy'][-1]
val_acc_final   = history.history['val_accuracy'][-1]
gap = train_acc_final - val_acc_final

print(f'Final train accuracy : {train_acc_final*100:.2f}%')
print(f'Final val   accuracy : {val_acc_final*100:.2f}%')
print(f'Train-val gap        : {gap*100:.2f}%  ', end='')
print('(healthy — model generalising)' if gap < 0.08 else '(check for overfit)')
print(f'\n📊 Test set accuracy : {test_acc*100:.2f}%')
print()
print('Reminder: a lower test accuracy than v2 (99%) is EXPECTED and CORRECT.')
print('The model is less memorised, more generalised. Real-world performance will be better.')


# ── Per-class accuracy check ────────────────────────────────────────────────
# This shows which specific characters are still being confused.
print('\n── Per-class accuracy (bottom 10 classes) ───────────────────────────')
all_preds, all_true = [], []
for xb, yb in test_ds:
    preds = model.predict(xb, verbose=0)
    all_preds.extend(np.argmax(preds, axis=1))
    all_true.extend(np.argmax(yb.numpy(), axis=1))

all_preds, all_true = np.array(all_preds), np.array(all_true)
class_accs = []
for cls_idx in range(NUM_CLASSES):
    mask = all_true == cls_idx
    if mask.sum() > 0:
        acc = (all_preds[mask] == cls_idx).mean()
        label = CLASS_MAPPING[str(cls_idx)]['label']
        class_accs.append((acc, cls_idx, label))

class_accs.sort()  # worst first
print(f'{"Idx":<5} {"Label":<20} {"Accuracy"}')
print('─' * 40)
for acc, idx, label in class_accs[:10]:
    bar = '█' * int(acc * 20)
    print(f'{idx:<5} {label:<20} {acc*100:>6.1f}%  {bar}')

## 🔬 Cell 11 — Inference Preprocessing Function

> **Unchanged from v2.** The preprocessing pipeline was working correctly.
> The problem was in the model, not the preprocessing.

Port this function 1:1 to C# OpenCV in your Unity `CameraManager.cs`.

In [ ]:
def preprocess_for_inference(image_bgr: np.ndarray) -> np.ndarray:
    """
    Convert any real-world BGR image to a model-ready (64, 64, 1) float32 tensor.
    Black background (~0), white strokes (~1). Character centred with safety padding.
    """
    if image_bgr.ndim == 2:
        gray = image_bgr
    else:
        gray = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY)

    # Step 1: CLAHE — boost local contrast
    clahe    = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    enhanced = clahe.apply(gray)

    # Step 2: Otsu binarization — adaptive ink/paper separation
    _, binary = cv2.threshold(enhanced, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    # Step 3: Auto-inversion — ensure white strokes on black background
    if np.mean(binary) > 128:
        binary = cv2.bitwise_not(binary)

    # Step 4: Bounding box crop + 15% safety padding
    coords = cv2.findNonZero(binary)
    if coords is not None:
        x, y, w, h = cv2.boundingRect(coords)
        pad = int(max(w, h) * 0.15)
        x1 = max(0, x - pad);               y1 = max(0, y - pad)
        x2 = min(binary.shape[1], x+w+pad); y2 = min(binary.shape[0], y+h+pad)
        cropped = binary[y1:y2, x1:x2]
    else:
        print('⚠️  No strokes found. Check Step 3 in the visualizer.')
        cropped = binary

    # Step 5: Aspect-ratio-preserving square padding
    ch, cw  = cropped.shape
    diff    = abs(ch - cw); half, r = diff // 2, diff % 2
    if   ch > cw: padded = cv2.copyMakeBorder(cropped, 0, 0, half, half+r, cv2.BORDER_CONSTANT, value=0)
    elif cw > ch: padded = cv2.copyMakeBorder(cropped, half, half+r, 0, 0, cv2.BORDER_CONSTANT, value=0)
    else:         padded = cropped

    # Step 6: INTER_AREA resize + final Otsu to remove blur
    resized = cv2.resize(padded, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
    _, final = cv2.threshold(resized, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    return (final.astype(np.float32) / 255.0)[:, :, np.newaxis]  # (64, 64, 1)


print('✅ preprocess_for_inference() defined (unchanged from v2)')

## 🖼️ Cell 12 — Preprocessing Visualizer (unchanged from v2)

Upload a real handwritten character image and verify all 6 steps.
**Step ④ (red border) must show white strokes on black background.**

In [ ]:
def visualize_preprocessing_steps(image_path: str) -> None:
    raw_bgr = cv2.imread(image_path)
    if raw_bgr is None:
        raise FileNotFoundError(f'Cannot read: {image_path}')

    raw_rgb  = cv2.cvtColor(raw_bgr, cv2.COLOR_BGR2RGB)
    gray     = cv2.cvtColor(raw_bgr, cv2.COLOR_BGR2GRAY) if raw_bgr.ndim == 3 else raw_bgr
    clahe    = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    enhanced = clahe.apply(gray)
    _, binary   = cv2.threshold(enhanced, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    inverted    = cv2.bitwise_not(binary) if np.mean(binary) > 128 else binary.copy()

    coords = cv2.findNonZero(inverted)
    if coords is not None:
        x, y, w, h = cv2.boundingRect(coords)
        pad = int(max(w, h) * 0.15)
        x1, y1 = max(0, x-pad), max(0, y-pad)
        x2, y2 = min(inverted.shape[1], x+w+pad), min(inverted.shape[0], y+h+pad)
        cropped = inverted[y1:y2, x1:x2]
    else:
        cropped = inverted

    ch, cw  = cropped.shape; diff = abs(ch-cw); half, r = diff//2, diff%2
    if   ch > cw: padded = cv2.copyMakeBorder(cropped, 0, 0, half, half+r, cv2.BORDER_CONSTANT, value=0)
    elif cw > ch: padded = cv2.copyMakeBorder(cropped, half, half+r, 0, 0, cv2.BORDER_CONSTANT, value=0)
    else:         padded = cropped

    resized = cv2.resize(padded, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
    _, final = cv2.threshold(resized, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    panels = [raw_rgb, enhanced, binary, inverted, padded, final]
    titles = [
        f'① Raw Input\n{raw_bgr.shape[1]}×{raw_bgr.shape[0]}',
        '② Grayscale+CLAHE', '③ Otsu Binarized',
        '④ Auto-Inverted\n← strokes must be WHITE',
        '⑤ Cropped+Padded', f'⑥ Final {IMG_SIZE}×{IMG_SIZE}\n← model input'
    ]
    fig, axes = plt.subplots(1, 6, figsize=(22, 4))
    fig.suptitle('TrueRead — Inference Preprocessing Sanity Check',
                 fontsize=13, fontweight='bold', y=1.03)
    for ax, img, title in zip(axes, panels, titles):
        ax.imshow(img, cmap='gray' if img.ndim == 2 else None)
        ax.set_title(title, fontsize=8.5, pad=6, linespacing=1.4); ax.axis('off')
    for spine in axes[3].spines.values():
        spine.set_visible(True); spine.set_edgecolor('red'); spine.set_linewidth(2.5)

    plt.tight_layout()
    plt.savefig('preprocessing_visualization.png', dpi=150, bbox_inches='tight')
    plt.show()

    white_pct = 100.0 * final.mean() / 255
    print(f'✅ White pixel %: {white_pct:.1f}%  (healthy: 5%–35%)')
    if white_pct > 50: print('⚠️  Over 50% white — Step ④ may be inverted.')
    if white_pct < 2:  print('⚠️  Under 2% white — very few strokes detected.')


from google.colab import files as colab_files
print('Upload a photo of a handwritten Hindi character:')
uploaded = colab_files.upload()
if uploaded:
    visualize_preprocessing_steps(list(uploaded.keys())[0])

## 🎯 Cell 13 — Single-Image Inference with Confidence Gate

This is the v3 inference test. It includes a **confidence gate** —
if the model's top prediction is below `MIN_CONFIDENCE`, it reports `uncertain`
instead of showing a potentially wrong character.

**Use this cell to calibrate your `MIN_CONFIDENCE` value before setting it in Unity.**

In [ ]:
MIN_CONFIDENCE = 0.70  # Predictions below this are shown as 'uncertain'
                        # Tune this value based on what you observe below

from google.colab import files as colab_files
print('Upload a photo of a handwritten Hindi character:')
uploaded = colab_files.upload()

if uploaded:
    img_path  = list(uploaded.keys())[0]
    raw_bgr   = cv2.imread(img_path)
    tensor    = preprocess_for_inference(raw_bgr)       # (64, 64, 1)
    batch     = np.expand_dims(tensor, axis=0)          # (1, 64, 64, 1)
    probs     = model.predict(batch, verbose=0)[0]      # (46,)
    top3_idxs = np.argsort(probs)[::-1][:3]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
    fig.suptitle('TrueRead v3 — Inference Result', fontsize=13, fontweight='bold')
    ax1.imshow(cv2.cvtColor(raw_bgr, cv2.COLOR_BGR2RGB))
    ax1.set_title('Your Input', fontsize=10); ax1.axis('off')
    ax2.imshow(tensor[:,:,0], cmap='gray')
    ax2.set_title('Preprocessed (fed to model)', fontsize=10); ax2.axis('off')
    plt.tight_layout(); plt.show()

    top_conf  = probs[top3_idxs[0]]
    top_label = CLASS_MAPPING[str(top3_idxs[0])]['label']

    print('\n' + '─' * 58)
    print(f'{"Rank":<6} {"Label":<20} {"Confidence":<12} Bar')
    print('─' * 58)
    for rank, idx in enumerate(top3_idxs, 1):
        label = CLASS_MAPPING[str(idx)]['label']
        conf  = probs[idx] * 100
        bar   = '█' * int(conf / 2.5)
        flag  = '  ← RESULT' if rank == 1 and probs[idx] >= MIN_CONFIDENCE else ''
        print(f'#{rank:<5} {label:<20} {conf:>6.2f}%     {bar}{flag}')
    print('─' * 58)

    if top_conf >= MIN_CONFIDENCE:
        print(f'\n✅ Result: "{top_label}"  ({top_conf*100:.1f}% confidence)')
        print(f'   Above gate ({MIN_CONFIDENCE*100:.0f}%) — Unity will display this character.')
    else:
        print(f'\n⚠️  Result: UNCERTAIN  ({top_conf*100:.1f}% — below gate of {MIN_CONFIDENCE*100:.0f}%)')
        print(f'   Unity will show "?" or "Hold steady" instead of a wrong character.')
        print(f'   Best guess was "{top_label}" — but not confident enough to display.')

## 🎮 Cell 14 — Unity Confidence Threshold (C# Snippet)

The cell below prints the exact C# code to add to your `SentisInferenceManager.cs`.
This is the Unity-side fix that prevents wrong characters from being displayed.

In [ ]:
unity_code = '''
// ─────────────────────────────────────────────────────────────────────────
// ADD THIS to your SentisInferenceManager.cs — Confidence Gate
// ─────────────────────────────────────────────────────────────────────────

// Minimum confidence required to accept a prediction.
// If the model is less certain than this, return -1 (unknown) instead of
// displaying a wrong character.
// Start with 0.70f — tune down if genuine characters are being rejected.
[Range(0.5f, 0.99f)]
public float minConfidenceThreshold = 0.70f;


// Replace your existing prediction method with this version:
public (int classIndex, float confidence) RunInference(Texture2D inputTexture)
{
    // ... your existing preprocessing and model.Execute() code stays here ...

    // Get output tensor
    var outputTensor = engine.PeekOutput(outputName) as TensorFloat;
    outputTensor.CompleteOperationsAndDownload();

    // Find top prediction and its confidence
    int   bestIndex      = 0;
    float bestConfidence = 0f;

    for (int i = 0; i < outputTensor.shape[1]; i++)
    {
        float conf = outputTensor[0, i];
        if (conf > bestConfidence)
        {
            bestConfidence = conf;
            bestIndex      = i;
        }
    }

    // ── Confidence Gate ──────────────────────────────────────────────────
    // This is the fix for wrong predictions being displayed.
    // If the model is not confident enough, return -1.
    // Your caller should check: if (classIndex == -1) { show "Hold steady"; }
    if (bestConfidence < minConfidenceThreshold)
    {
        Debug.Log($"[TrueRead] Low confidence: {bestConfidence:P0} — rejecting prediction");
        return (-1, bestConfidence);
    }

    Debug.Log($"[TrueRead] Predicted class {bestIndex} with {bestConfidence:P0} confidence");
    return (bestIndex, bestConfidence);
}


// ── In your UI update code (wherever you display the result): ────────────
var (classIndex, confidence) = inferenceManager.RunInference(capturedTexture);

if (classIndex == -1)
{
    resultText.text = "Hold steady...";
    confidenceBar.value = confidence;  // still show low bar
}
else
{
    var charData = characterDatabase.GetByIndex(classIndex);
    resultText.text = charData.devanagariChar;
    confidenceBar.value = confidence;
}
// ─────────────────────────────────────────────────────────────────────────
'''

print(unity_code)

## 📱 Cell 15 — TFLite Export

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_bytes = converter.convert()

TFLITE_PATH = 'trueread_v3_model.tflite'
with open(TFLITE_PATH, 'wb') as f:
    f.write(tflite_bytes)

size_mb = os.path.getsize(TFLITE_PATH) / (1024*1024)
print(f'✅ {TFLITE_PATH}  ({size_mb:.2f} MB)')

interp = tf.lite.Interpreter(model_path=TFLITE_PATH)
interp.allocate_tensors()
inp, out = interp.get_input_details()[0], interp.get_output_details()[0]
print(f'   Input  shape: {inp["shape"]}  dtype: {inp["dtype"]}')
print(f'   Output shape: {out["shape"]}  dtype: {out["dtype"]}')

## 🔄 Cell 16 — ONNX Export (for Unity Sentis)
> ⚠️ **Screenshot the tensor names printed below** — you need them in `SentisInferenceManager.cs`.

In [ ]:
import tf2onnx, onnx

ONNX_PATH = 'trueread_v3_model.onnx'

input_sig = [tf.TensorSpec(
    shape=[None, IMG_SIZE, IMG_SIZE, 1], dtype=tf.float32, name='input'
)]

model_proto, _ = tf2onnx.convert.from_keras(
    model, input_signature=input_sig, opset=13, output_path=ONNX_PATH
)

onnx_mb    = os.path.getsize(ONNX_PATH) / (1024*1024)
onnx_model = onnx.load(ONNX_PATH)
onnx.checker.check_model(onnx_model)
inp_name   = onnx_model.graph.input[0].name
out_name   = onnx_model.graph.output[0].name

print(f'✅ {ONNX_PATH}  ({onnx_mb:.2f} MB)')
print(f'   Validation: passed  |  Opset: {onnx_model.opset_import[0].version}')
print()
print('═' * 55)
print(' ⚠️  Copy these into SentisInferenceManager.cs:')
print(f'   Input  tensor name : "{inp_name}"')
print(f'   Output tensor name : "{out_name}"')
print('═' * 55)

## 💾 Cell 17 — Save All Outputs to Google Drive

In [ ]:
OUTPUT_DIR = os.path.join(DRIVE_DATASET_PATH, 'model_outputs_v3')
os.makedirs(OUTPUT_DIR, exist_ok=True)

FILES = [
    ('trueread_v3_model.onnx',   'Unity: Assets/Models/'),
    ('trueread_v3_model.tflite', 'Backup/mobile testing'),
    ('class_mapping.json',       'Unity: Assets/StreamingAssets/'),
    ('training_curves.png',      'Reference'),
    ('preprocessing_visualization.png', 'Reference'),
]

print(f'Saving to: {OUTPUT_DIR}\n')
for fname, dest in FILES:
    src = fname if os.path.exists(fname) else os.path.join(DRIVE_DATASET_PATH, fname)
    if os.path.exists(src):
        shutil.copy(src, os.path.join(OUTPUT_DIR, fname))
        kb = os.path.getsize(os.path.join(OUTPUT_DIR, fname)) / 1024
        print(f'  ✅ {fname:<45} {kb:>8.1f} KB  →  {dest}')
    else:
        print(f'  ⚠️  {fname} — not found, skipped')

print(f'\n🎉 Done!')
print('\nUnity checklist:')
print('  1. trueread_v3_model.onnx  →  Assets/Models/')
print('  2. class_mapping.json      →  Assets/StreamingAssets/')
print('  3. Tensor names from Cell 16 → SentisInferenceManager.cs')
print('  4. Confidence gate from Cell 14 → SentisInferenceManager.cs')
print('  5. Set minConfidenceThreshold = 0.70f (tune if needed)')